In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.default.bronze_country_year_summary")
display(bronze_df.limit(5))

#null check
null_check = bronze_df.select([count(when(isnull(c), c)).alias(c) for c in bronze_df.columns])
display(null_check)

#check duplicate records
duplicates_check = bronze_df.groupBy("country","year").count().filter(col("count")>1)
display(duplicates_check)

In [0]:
#Screen Time Category

silver_df = bronze_df.withColumn("screen_time_category",when(col("avg_screen_time_hours") < 3, "Low")
    .when(col("avg_screen_time_hours") < 5, "Moderate")
    .when(col("avg_screen_time_hours") < 7, "High")
    .otherwise("Very High"))

display(silver_df.limit(5))


In [0]:
# Sleep Category

silver_df = silver_df.withColumn(
    "sleep_category",
    when(col("avg_sleep_hours") >= 8, "Healthy Sleep")
    .when(col("avg_sleep_hours") >= 6, "Average Sleep")
    .otherwise("Poor Sleep")
)

display(silver_df.limit(5))


In [0]:
# Mental Health Risk Category
silver_df = silver_df.withColumn(
    "risk_category",
    when(col("avg_mental_health_risk")<30, 'Low Risk')
    .when(col("avg_mental_health_risk")<60, 'Moderate Risk')
    .otherwise('High Risk')
)

display(silver_df.limit(5))


In [0]:
#Digital Wellness Score Band

silver_df = silver_df.withColumn(
    "wellbeing_band",
    when(col("avg_digital_wellbeing") >= 70, "Excellent")
    .when(col("avg_digital_wellbeing") >= 50, "Good")
    .when(col("avg_digital_wellbeing") >= 30, "Average")
    .otherwise("Poor")
)

display(silver_df.limit(5))

In [0]:
#Social Media Dependency Index

silver_df = silver_df.withColumn(
    "social_media_dependency",
    round(
        col("avg_social_media_hours")
        / col("avg_screen_time_hours"),
        2
    )
)

display(silver_df.limit(5))

In [0]:
#Wellness Gap

silver_df = silver_df.withColumn(
    "wellness_gap",
    round(
        col("avg_digital_wellbeing")
        - col("avg_mental_health_risk"),
        2
    )
)

display(silver_df.limit(5))

In [0]:
#checking new columns

display(
    silver_df.select(
        "country",
        "year",
        "screen_time_category",
        "sleep_category",
        "risk_category",
        "wellbeing_band",
        "social_media_dependency",
        "wellness_gap"
    )
)

#save the silver table
silver_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_digital_wellbeing")